# Bab 4 (4.5 - 4.7): Membangun Data IoT Sintetis dengan Python

Buku: *IoT Terintegrasi dengan Machine Learning Data Analytics* (Onno W. Purbo), Bab 4 bagian 4.5 sampai 4.7, yang berada di antara Hands-on 8 dan Hands-on 9. Setiap sel kode sama dengan skrip pada folder `scripts/`.

Jalankan dari root repo atau dari folder `notebooks/` (sel pertama mengatur direktori kerja).

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(Path.cwd())

/downloads/iot-ml-book


## 4.5.1 Contoh 1: suhu dan kelembaban harian (`scripts/generate_room_synthetic.py`)

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_PATH = Path("data/synthetic/room_daily_synthetic.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Waktu 1 hari, interval 10 menit
timestamps = pd.date_range(
    start="2026-01-01 00:00:00",
    end="2026-01-01 23:50:00",
    freq="10min"
)

hours = timestamps.hour + timestamps.minute / 60.0

# Pola suhu harian: rendah malam, naik siang
temperature = 24 + 4 * np.sin((hours - 6) / 24 * 2 * np.pi)
temperature += np.random.normal(0, 0.3, len(timestamps))

# kelembaban berlawanan dengan suhu
humidity = 70 - 10 * np.sin((hours - 6) / 24 * 2 * np.pi)
humidity += np.random.normal(0, 1.5, len(timestamps))

df = pd.DataFrame({
    "timestamp": timestamps,
    "device_id": "room01",
    "temperature_c": np.round(temperature, 2),
    "humidity_pct": np.round(humidity, 2)
})

df.to_csv(OUTPUT_PATH, index=False)
print(f"Data sintetik disimpan ke: {OUTPUT_PATH}")
print(df.head())

Data sintetik disimpan ke: data/synthetic/room_daily_synthetic.csv
            timestamp device_id  temperature_c  humidity_pct
0 2026-01-01 00:00:00    room01          19.63         81.89
1 2026-01-01 00:10:00    room01          19.75         80.79
2 2026-01-01 00:20:00    room01          19.23         78.53
3 2026-01-01 00:30:00    room01          19.90         81.89
4 2026-01-01 00:40:00    room01          20.06         81.61


## 4.5.2 Contoh 2: konsumsi energi dengan pola jam sibuk (`scripts/generate_energy_synthetic.py`)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_PATH = Path("data/synthetic/energy_daily_synthetic.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

timestamps = pd.date_range(
    start="2026-01-01 00:00:00",
    end="2026-01-01 23:50:00",
    freq="10min"
)

power = []

for ts in timestamps:
    hour = ts.hour

    if 0 <= hour < 6:
        base = 120
    elif 6 <= hour < 8:
        base = 180
    elif 8 <= hour < 12:
        base = 350
    elif 12 <= hour < 14:
        base = 450
    elif 14 <= hour < 18:
        base = 320
    else:
        base = 170

    value = base + np.random.normal(0, 15)
    power.append(value)

df = pd.DataFrame({
    "timestamp": timestamps,
    "device_id": "lab-energy-01",
    "power_watt": np.round(power, 2)
})

df.to_csv(OUTPUT_PATH, index=False)
print(f"Data energi sintetik disimpan ke: {OUTPUT_PATH}")
print(df.head())

Data energi sintetik disimpan ke: data/synthetic/energy_daily_synthetic.csv
            timestamp      device_id  power_watt
0 2026-01-01 00:00:00  lab-energy-01      127.60
1 2026-01-01 00:10:00  lab-energy-01      106.53
2 2026-01-01 00:20:00  lab-energy-01      139.66
3 2026-01-01 00:30:00  lab-energy-01      119.18
4 2026-01-01 00:40:00  lab-energy-01      115.61


## 4.5.3 Contoh 3: getaran mesin normal vs fault (`scripts/generate_vibration_synthetic.py`)

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_PATH = Path("data/synthetic/vibration_machine_synthetic.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

n = 1000
t = np.arange(n)

# Sinyal normal
normal_signal = 0.8 * np.sin(2 * np.pi * 0.03 * t) + np.random.normal(0, 0.1, n)

# Sinyal fault
fault_signal = 1.6 * np.sin(2 * np.pi * 0.03 * t) + np.random.normal(0, 0.25, n)

df_normal = pd.DataFrame({
    "sample_id": range(n),
    "machine_id": "motor01",
    "vibration": np.round(normal_signal, 4),
    "label": "normal"
})

df_fault = pd.DataFrame({
    "sample_id": range(n, 2*n),
    "machine_id": "motor01",
    "vibration": np.round(fault_signal, 4),
    "label": "fault"
})

df = pd.concat([df_normal, df_fault], ignore_index=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Data getaran sintetik disimpan ke: {OUTPUT_PATH}")
print(df.head())
print(df.tail())

Data getaran sintetik disimpan ke: data/synthetic/vibration_machine_synthetic.csv
   sample_id machine_id  vibration   label
0          0    motor01    -0.0075  normal
1          1    motor01     0.1797  normal
2          2    motor01     0.1754  normal
3          3    motor01     0.3927  normal
4          4    motor01     0.5160  normal
      sample_id machine_id  vibration  label
1995       1995    motor01    -1.7822  fault
1996       1996    motor01    -1.0837  fault
1997       1997    motor01    -0.8315  fault
1998       1998    motor01    -0.5124  fault
1999       1999    motor01    -0.1768  fault


## 4.5.4 Menambahkan noise pada data sensor

In [5]:
import numpy as np

clean_temp = np.array([25.0, 25.2, 25.4, 25.3, 25.5])
noisy_temp = clean_temp + np.random.normal(0, 0.2, len(clean_temp))

print("Data asli :", clean_temp)
print("Data noisy:", np.round(noisy_temp, 2))

Data asli : [25.  25.2 25.4 25.3 25.5]
Data noisy: [25.12 24.87 25.37 25.54 25.47]


## 4.5.5 Missing value, drift, dan outlier buatan (`scripts/inject_data_issues.py`)

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_PATH = Path("data/synthetic/room_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/room_daily_synthetic_corrupted.csv")

df = pd.read_csv(INPUT_PATH)

# 1. Missing value acak
missing_idx = np.random.choice(df.index, size=8, replace=False)
df.loc[missing_idx, "temperature_c"] = np.nan

# 2. Drift bertahap pada suhu
df["temperature_c"] = df["temperature_c"] + np.linspace(0, 1.2, len(df))

# 3. Outlier buatan
outlier_idx = np.random.choice(df.index, size=3, replace=False)
df.loc[outlier_idx, "temperature_c"] = df.loc[outlier_idx, "temperature_c"] + 8

df.to_csv(OUTPUT_PATH, index=False)
print(f"Data dengan missing value, drift, dan outlier disimpan ke: {OUTPUT_PATH}")
print("Index missing:", missing_idx)
print("Index outlier:", outlier_idx)

Data dengan missing value, drift, dan outlier disimpan ke: data/synthetic/room_daily_synthetic_corrupted.csv
Index missing: [ 71  60 135 141  13  53  39 108]
Index outlier: [140  27 133]


## 4.5.6 Contoh 1: label nyaman / tidak nyaman (`scripts/label_room_comfort_daily.py`)

In [7]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data/synthetic/room_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/room_daily_labeled.csv")

df = pd.read_csv(INPUT_PATH)

df["comfort_label"] = df.apply(
    lambda row: "nyaman"
    if (24 <= row["temperature_c"] <= 28 and 45 <= row["humidity_pct"] <= 65)
    else "tidak_nyaman",
    axis=1
)

df.to_csv(OUTPUT_PATH, index=False)
print(df.head())

             timestamp device_id  temperature_c  humidity_pct comfort_label
0  2026-01-01 00:00:00    room01          19.63         81.89  tidak_nyaman
1  2026-01-01 00:10:00    room01          19.75         80.79  tidak_nyaman
2  2026-01-01 00:20:00    room01          19.23         78.53  tidak_nyaman
3  2026-01-01 00:30:00    room01          19.90         81.89  tidak_nyaman
4  2026-01-01 00:40:00    room01          20.06         81.61  tidak_nyaman


## 4.5.6 Contoh 2: label normal / warning / fault (`scripts/label_energy_status_daily.py`)

In [8]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data/synthetic/energy_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/energy_daily_labeled.csv")

df = pd.read_csv(INPUT_PATH)

def label_energy(power):
    if power < 250:
        return "normal"
    elif power < 400:
        return "warning"
    return "fault"

df["status_label"] = df["power_watt"].apply(label_energy)
df.to_csv(OUTPUT_PATH, index=False)
print(df.head())

             timestamp      device_id  power_watt status_label
0  2026-01-01 00:00:00  lab-energy-01      127.60       normal
1  2026-01-01 00:10:00  lab-energy-01      106.53       normal
2  2026-01-01 00:20:00  lab-energy-01      139.66       normal
3  2026-01-01 00:30:00  lab-energy-01      119.18       normal
4  2026-01-01 00:40:00  lab-energy-01      115.61       normal


## 4.5.6 Contoh 3: label hemat / boros (`scripts/label_energy_efficiency_daily.py`)

In [9]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data/synthetic/energy_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/energy_daily_efficiency.csv")

df = pd.read_csv(INPUT_PATH)

mean_power = df["power_watt"].mean()

df["efficiency_label"] = df["power_watt"].apply(
    lambda x: "hemat" if x <= mean_power else "boros"
)

df.to_csv(OUTPUT_PATH, index=False)
print(df.head())

             timestamp      device_id  power_watt efficiency_label
0  2026-01-01 00:00:00  lab-energy-01      127.60            hemat
1  2026-01-01 00:10:00  lab-energy-01      106.53            hemat
2  2026-01-01 00:20:00  lab-energy-01      139.66            hemat
3  2026-01-01 00:30:00  lab-energy-01      119.18            hemat
4  2026-01-01 00:40:00  lab-energy-01      115.61            hemat


## 4.7 Validasi sederhana (`scripts/validate_synthetic_room.py`)

In [10]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/synthetic/room_daily_synthetic.csv")
df = pd.read_csv(DATA_PATH)

print("=== Statistik dasar ===")
print(df.describe())

print("\n=== Korelasi ===")
print(df[["temperature_c", "humidity_pct"]].corr())

print("\n=== Cek nilai ekstrem ===")
print("Suhu min:", df["temperature_c"].min())
print("Suhu max:", df["temperature_c"].max())
print("Humidity min:", df["humidity_pct"].min())
print("Humidity max:", df["humidity_pct"].max())

=== Statistik dasar ===
       temperature_c  humidity_pct
count     144.000000    144.000000
mean       23.998889     70.151319
std         2.860357      7.271572
min        19.230000     57.380000
25%        21.245000     63.240000
50%        23.855000     69.970000
75%        26.687500     77.262500
max        28.300000     82.310000

=== Korelasi ===
               temperature_c  humidity_pct
temperature_c       1.000000     -0.973983
humidity_pct       -0.973983      1.000000

=== Cek nilai ekstrem ===
Suhu min: 19.23
Suhu max: 28.3
Humidity min: 57.38
Humidity max: 82.31


## Penutup

Hasil running: dataset sintetik ruangan (interval 10 menit), energi harian, dan getaran normal/fault berhasil dibuat lalu divalidasi. Rentang suhu hasil generate 19.23-28.3 C dengan korelasi suhu-kelembaban -0.97; variasi noise, missing value, drift, dan outlier tersimpan di `data/synthetic/`.
